# L2: Constructing the Memory Manager

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

</div>

## Part 1: Setting Up Database, Vector Stores and Embedding Models

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from helper import suppress_warnings

suppress_warnings()

from helper import connect_to_postgres

# Connect as the VECTOR user for all subsequent operations
database_connection = connect_to_postgres()

2026-04-25 23:07:14.149 | INFO     | helper:connect_to_postgres:33 - Connection attempt 1/3...
2026-04-25 23:07:14.206 | INFO     | helper:connect_to_postgres:45 - Connected successfully!
2026-04-25 23:07:14.237 | INFO     | helper:connect_to_postgres:50 - DB version: PostgreSQL 18.3 (Debian 18.3-1.pgdg12+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


### Loading the Embedding Model

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-mpnet-base-v2"
)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1699.45it/s]


### Define Memory Tables and Stores
First, we define table names for each memory type. 

These tables will be created in Oracle Database to persist agent memory.

### Memory Types We'll Implement

| Memory Type | Human Analogy | Purpose | Storage | Retrieval Strategies Used |
|-------------|---------------|---------|---------|---------------------------|
| **Conversational** | Short-term memory | Chat history per thread | SQL Table | Exact match by thread_id |
| **Knowledge Base** | Long-term semantic memory | Facts, documents, search results | Vector Store | Semantic similarity search |
| **Workflow** | Procedural memory | Learned action patterns | Vector Store | Semantic similarity search + metadata filtering |
| **Toolbox** | Skill memory | Available tools & capabilities | Vector Store | Semantic similarity search |
| **Entity** | Episodic memory | People, places, systems mentioned | Vector Store | Semantic similarity search |
| **Summary** | Compressed memory | Condensed context for long conversations | Vector Store | Semantic similarity search (with optional ID filter) |
| **Tool Log** | Execution audit trail | Raw tool inputs/outputs and execution status | SQL Table | Exact match by thread_id + timestamp ordering |


### Create Vector Stores for Each Memory Type

Here we create 5 separate vector stores—one for each memory type. 

Each vector store is backed by its own Oracle table and uses the same embedding model for consistency.

| Vector Store | Purpose |
|--------------|---------|
| `knowledge_base_vs` | Store documents, facts, and search results |
| `workflow_vs` | Store learned action patterns and tool sequences |
| `toolbox_vs` | Store tool definitions for semantic tool discovery |
| `entity_vs` | Store extracted entities (people, places, systems) |
| `summary_vs` | Store compressed summaries for long conversations |


In [17]:
from langchain_community.vectorstores import PGVector

from src.config.settings import settings

class StoreManager:
    """Manages vector stores and SQL tables."""

    def __init__(self, conn, embedding_function, table_names, distance_strategy,
                 conversational_table, tool_log_table: str | None = None):
        """Initialize all stores."""
        self.conn = conn
        self.embedding_function = embedding_function
        self.distance_strategy = distance_strategy
        self._conversational_table = conversational_table
        self._tool_log_table = tool_log_table

        connection_string = PGVector.connection_string_from_db_params(
            driver="psycopg2",
            user=settings.POSTGRES_USER,
            password=settings.POSTGRES_PASSWORD,
            host=settings.POSTGRES_HOST,
            port=settings.POSTGRES_PORT,
            database=settings.POSTGRES_DB,
        )

        self._knowledge_base_vs = PGVector(
            embedding_function=embedding_function,
            collection_name=table_names['knowledge_base'],
            connection_string=connection_string,
            distance_strategy=distance_strategy,
        )

        self._workflow_vs = PGVector(
            embedding_function=embedding_function,
            collection_name=table_names['workflow'],
            connection_string=connection_string,
            distance_strategy=distance_strategy,
        )

        self._toolbox_vs = PGVector(
            embedding_function=embedding_function,
            collection_name=table_names['toolbox'],
            connection_string=connection_string,
            distance_strategy=distance_strategy,
        )

        self._entity_vs = PGVector(
            embedding_function=embedding_function,
            collection_name=table_names['entity'],
            connection_string=connection_string,
            distance_strategy=distance_strategy,
        )

        self._summary_vs = PGVector(
            embedding_function=embedding_function,
            collection_name=table_names['summary'],
            connection_string=connection_string,
            distance_strategy=distance_strategy,
        )

    def get_knowledge_base_store(self):
        """Return the knowledge base vector store."""
        return self._knowledge_base_vs

    def get_workflow_store(self):
        """Return the workflow vector store."""
        return self._workflow_vs

    def get_toolbox_store(self):
        """Return the toolbox vector store."""
        return self._toolbox_vs

    def get_entity_store(self):
        """Return the entity vector store."""
        return self._entity_vs

    def get_summary_store(self):
        """Return the summary vector store."""
        return self._summary_vs

    def get_conversational_table(self):
        """Return the conversational history table name."""
        return self._conversational_table

    def get_tool_log_table(self):
        """Return the tool log table name."""
        return self._tool_log_table
    
    def set_up_hybrid_search(self, preference_name=""):
        pass


In [18]:
# Create StoreManager instance
store_manager = StoreManager(
    conn=database_connection,
    embedding_function=embedding_model,
    table_names={
        'knowledge_base': KNOWLEDGE_BASE_TABLE,
        'workflow': WORKFLOW_TABLE,
        'toolbox': TOOLBOX_TABLE,
        'entity': ENTITY_TABLE,
        'summary': SUMMARY_TABLE,
    },
    distance_strategy="cosine",
    conversational_table="conversational_memory",
    tool_log_table="tool_log_memory",
)

In [19]:
# Get all stores via the manager
conversation_table = store_manager.get_conversational_table()
knowledge_base_vs = store_manager.get_knowledge_base_store()
workflow_vs = store_manager.get_workflow_store()
toolbox_vs = store_manager.get_toolbox_store()
entity_vs = store_manager.get_entity_store()
summary_vs = store_manager.get_summary_store()
tool_log_table = store_manager.get_tool_log_table()

### Classification of Memory operation in Agentic Systems

A key design decision in memory engineering is determining which operations should be **Deterministic** (executed automatically by code) versus **Agent-Triggered** (decided by the LLM at runtime).

- A **deterministic** memory operation is one that runs based on system rules, not the model’s discretion. It is executed every time (or under clearly defined, non-negotiable conditions) so the system behaves predictably.
- An **agent-triggered** memory operation runs only when the model decides it’s necessary, based on intent and situation.

| Operation | Deterministic | Agent-Triggered |
|-----------|:------------:|:-------:|
| `read_conversational_memory()` | ✅ | ❌ |
| `read_knowledge_base()` | ✅ | ❌ |
| `read_workflow()` | ✅ | ❌ |
| `read_entity()` | ✅ | ❌ |
| `read_summary_context()` | ❌ | ✅ |
| `write_conversational_memory()` | ✅ | ❌ |
| `write_workflow()` | ✅ | ❌ |
| `write_entity()` | ❌ | ✅ |
| `search_tavily()` | ❌ | ✅ |
| `expand_summary()` | ❌ | ✅ |
| `summarize_and_store()` | ❌ | ✅ |
| `read_toolbox()` | ✅ | ✅ |


Deterministic memory operations run:
- **every turn**, or
- under **explicit, fixed conditions** (e.g., “always at the start of the agent loop”, “always after tool execution”)

### Why Deterministic Retrieval Is Useful
Memory retrieval is commonly run **at the start of each agent loop** because:

1. **Context bootstrapping is non-negotiable**
   - The agent needs prior context to remain consistent and avoid repeating mistakes.
   - Without deterministic retrieval, the agent behaves “stateless” and starts from scratch.

2. **The agent can’t choose to look up what it doesn’t know exists**
   - If the agent must decide whether to check memory, it must guess what’s stored.
   - This creates a chicken-and-egg problem: *you need memory to know which memory you need.*

3. **Predictability**
   - Always loading memory produces consistent behavior and makes the system easier to evaluate and debug.

### Why Deterministic Storage Is Useful
Persisting conversations, workflows, and entities is often deterministic because:

1. **Reliability**
   - You don’t want the agent to “forget to save” important information.
   - If continuity matters, persistence must be consistent.

2. **Completeness**
   - Every interaction should be recorded to avoid gaps.
   - Selective saving creates missing context that later breaks long-horizon tasks.

3. **Reduced cognitive load**
   - The model should focus on task execution, not memory bookkeeping.

### Advantages of Deterministic Memory Operations
- **Predictable behavior** across runs and turns
- **Stronger continuity** (fewer “stateless resets”)
- **Fewer missed memories** (higher reliability)
- **Easier debugging and evaluation** (clear expectations of what should be loaded/saved)


This is appropriate for memory actions that require judgment, such as:
- “Should this be saved as a durable preference?”
- “Should I consolidate/summarize now?”
- “Do I need deeper retrieval beyond the baseline preload?”
- “Should I strengthen, update, merge, or decay this memory?”

### Why Agent-Triggered Memory Operations Are Useful

1. **Relevance**
   - Not everything deserves long-term storage.
   - The agent can distinguish signal (preferences, decisions, constraints) from noise.

2. **Cost and latency control**
   - Deep retrieval, reranking, summarization, and consolidation cost tokens/time.
   - Triggering only when needed reduces overhead.

3. **Higher-quality memory management**
   - Decisions about *what to store* and *how to compress* require semantic understanding of intent.
   - The model is well-suited to decide when a memory action is worthwhile.

### Advantages of Agent-Triggered Memory Operations
- **Higher signal-to-noise memory** (less clutter)
- **Reduced memory bloat**
- **Selective compute usage** (summarize/expand/retrieve only when valuable)
- **More human-like remembering** (store/retrieve when it matters)

### How Tool Calls Fit In

External tool calls (e.g., web search, external DB lookups, expensive summarization jobs) are typically **agent-triggered** because:

1. **Intent matters**
   - Only the agent can judge whether extra information is needed.
   - Automatically using tools for every query is wasteful.

2. **Cost considerations**
   - Tools often introduce latency and may incur API costs.
   - The agent should call tools only when the expected value is high.

3. **Judgment required**
   - Choosing *what* to search for or *what* to expand requires understanding the user’s goal.

---

## Part 2: Memory Manager Initialization

The `MemoryManager` class is the central abstraction that unifies all memory operations. It provides a clean interface for reading and writing to different memory types, hiding the complexity of SQL queries and vector store operations. It is a single class that manages 7 types of memory with consistent read/write patterns:

| Memory Type | Storage | Write Method | Read Method |
|-------------|---------|--------------|-------------|
| **Conversational** | SQL Table | `write_conversational_memory()` | `read_conversational_memory()` |
| **Knowledge Base** | Vector Store | `write_knowledge_base()` | `read_knowledge_base()` |
| **Workflow** | Vector Store | `write_workflow()` | `read_workflow()` |
| **Toolbox** | Vector Store | `write_toolbox()` | `read_toolbox()` |
| **Entity** | Vector Store | `write_entity()` | `read_entity()` |
| **Summary** | Vector Store | `write_summary()` | `read_summary_memory()`, `read_summary_context()` |
| **Tool Log** | SQL Table | `write_tool_log()` | `read_tool_logs()` |


In [22]:
from helper import MemoryManager

# Initialize the MemoryManager instance
# Note: Uses SQL table for conversational memory, vector stores for others
memory_manager = MemoryManager(
    conn=database_connection,
    conversation_table="conversational_memory",
    knowledge_base_vs=knowledge_base_vs,
    workflow_vs=workflow_vs,
    toolbox_vs=toolbox_vs,
    entity_vs=entity_vs,
    summary_vs=summary_vs,
    tool_log_table="tool_log_memory"
)

## Part 3: Using The Memory Manager

Ingest the knowledge base for **ArxivScout** from HuggingFace by streaming arXiv paper records from the `nick007x/arxiv-papers dataset`.

In [23]:
from datasets import load_dataset
from itertools import islice

ds = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

extracting only the key fields (title, subjects, abstract, submission date, and arXiv ID), concatenating title + subjects + abstract into a single searchable text payload, and writing each entry into memory_manager.write_knowledge_base(...) with the extracted fields stored as metadata for filtering and attribution.

In [24]:
for paper in islice(ds, 100):
    # extract the key fields
    title = (paper.get("title") or "").strip()
    abstract = (paper.get("abstract") or "").strip()
    subjects = (paper.get("subjects") or paper.get("primary_subject") or "").strip()
    submission_date = (paper.get("submission_date") or "").strip()

    # skip empty records
    if not (title or abstract or subjects):
        continue

    # concatenate the key fields containing context for semantic search
    text = "\n".join([part for part in (title, subjects, abstract) if part])

    memory_manager.write_knowledge_base(
        text=text,
        metadata={
            "arxiv_id": paper.get("arxiv_id"),
            "title": title,
            "subjects": subjects,
            "abstract": abstract,
            "submission_date": submission_date,
        },
    )

In [25]:
results = memory_manager.read_knowledge_base(query="space exploration")
print(results)

Source: {'arxiv_id': '0902.3000', 'title': 'A Census of Exoplanets in Orbits Beyond 0.5 AU via Space-based Microlensing', 'subjects': 'Earth and Planetary Astrophysics (astro-ph.EP); Astrophysics of Galaxies (astro-ph.GA)', 'abstract': 'A space-based gravitational microlensing exoplanet survey will provide a statistical census of exoplanets with masses greater than 0.1 Earth-masses and orbital separations ranging from 0.5AU to infinity. This includes analogs to all the Solar System&#39;s planets except for Mercury, as well as most types of planets predicted by planet formation theories. Such a survey will provide results on the frequency of planets around all types of stars except those with short lifetimes. Close-in planets with separations &lt; 0.5 AU are invisible to a space-based microlensing survey, but these can be found by Kepler. Other methods, including ground-based microlensing, cannot approach the comprehensive statistics on the mass and semi-major axis distribution of extra